# Batch Scoring — Readmission Risk Prediction

Scores every admission with the trained model to produce readmission-risk
predictions and a per-department risk summary.

**Reads:** `gold_ml_features` + saved model  **Writes:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count, isnan, udf,
    sum as spark_sum, round as spark_round
)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassificationModel

spark = SparkSession.builder.getOrCreate()
model = RandomForestClassificationModel.load('Files/models/readmission_rf')
df = spark.read.table('gold_ml_features')
print(f'Scoring {df.count():,} feature rows')

In [ ]:
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

numeric_features = [
    'length_of_stay_days', 'prior_admissions', 'admission_hour',
    'vital_count', 'abnormal_vital_count', 'abnormal_vital_ratio', 'avg_vital_value',
]
cat_cols = ['department', 'admission_type', 'insurance_type', 'age_group', 'dx_chapter']
indexed_df = df
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)

all_features = numeric_features + [f'{c}_idx' for c in cat_cols]
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(indexed_df)

In [ ]:
scored = model.transform(model_df)
extract_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())

predictions = (
    scored
    .withColumn('readmission_probability', spark_round(extract_prob(col('probability')), 4))
    .withColumn('predicted_readmission', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('readmission_probability') > 0.8, 'critical')
        .when(col('readmission_probability') > 0.6, 'high')
        .when(col('readmission_probability') > 0.4, 'medium')
        .otherwise('low'))
    .withColumn('scored_at', current_timestamp())
    .select(
        'admission_id', 'patient_id', 'department', 'admission_type',
        'insurance_type', 'age_group', 'los_bucket',
        'length_of_stay_days', 'abnormal_vital_count',
        'had_readmission', 'predicted_readmission', 'readmission_probability', 'risk_level',
        'scored_at')
)
predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per-department readmission risk summary
summary = (
    predictions
    .groupBy('department')
    .agg(
        count('*').alias('total_admissions'),
        spark_sum('predicted_readmission').alias('predicted_readmission_count'),
        spark_sum('had_readmission').alias('actual_readmission_count'),
        spark_round(avg('readmission_probability'), 4).alias('avg_readmission_probability'),
        spark_round(avg('length_of_stay_days'), 1).alias('avg_length_of_stay'),
        spark_round(avg('abnormal_vital_count'), 2).alias('avg_abnormal_vitals'),
    )
    .withColumn('readmission_rate', spark_round(col('predicted_readmission_count') / col('total_admissions') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_readmission_probability') > 0.6, 'high')
        .when(col('avg_readmission_probability') > 0.3, 'medium')
        .otherwise('low'))
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_readmission_probability').desc())
)
summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Department risk summary: {summary.count()} rows')
summary.show(15, truncate=False)

In [ ]:
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('All Gold ML tables optimized')